## 1. Загрузите выборку Boston с помощью функции `sklearn.datasets.load_boston()`.

Результатом вызова данной функции является объект, у которого признаки записаны в поле `data`, а целевой вектор — в поле `target`.

In [1]:
from sklearn import datasets
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import scale
import numpy as np
import pandas as pd

data_url = "http://lib.stat.cmu.edu/datasets/boston"
raw_df = pd.read_csv(data_url, sep="\s+", skiprows=22, header=None)

data = np.hstack([raw_df.values[::2, :], raw_df.values[1::2, :2]])
target = raw_df.values[1::2, 2]

features = data

## 2. Приведите признаки в выборке к одному масштабу при помощи функции `sklearn.preprocessing.scale`.

In [2]:
features_scaled = scale(features)

## 3. Переберите разные варианты параметра метрики `p` по сетке от 1 до 10 с таким шагом, чтобы всего было протестировано 200 вариантов (используйте функцию `numpy.linspace`).

Используйте `KNeighborsRegressor` с `n_neighbors=5` и `weights='distance'` — данный параметр добавляет в алгоритм веса, зависящие от расстояния до ближайших соседей. В качестве метрики качества используйте среднеквадратичную ошибку (параметр `scoring='neg_mean_squared_error'` у `cross_val_score`). Качество оценивайте с помощью кросс-валидации по 5 блокам с `random_state=42`, не забудьте включить перемешивание выборки (`shuffle=True`).

In [3]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
p_values = np.linspace(1, 10, 200)
mse_scores = []

for p in p_values:
    model = KNeighborsRegressor(n_neighbors=5, weights='distance', metric='minkowski', p=p)
    scores = cross_val_score(model, features_scaled, target, cv=kf, scoring='neg_mean_squared_error')
    mean_mse = -scores.mean()
    mse_scores.append(mean_mse)

## 4. Определите, при каком `p` качество на кросс-валидации оказалось оптимальным.

Обратите внимание, что `cross_val_score` возвращает массив показателей качества по блокам; необходимо максимизировать среднее этих показателей. Это значение параметра и будет ответом на задачу.

Если ответом является нецелое число, то целую и дробную часть необходимо разграничивать точкой, например, 0.4. При необходимости округляйте дробную часть до одного знака.

In [5]:
best_idx = np.argmin(mse_scores)
best_p = p_values[best_idx]
best_mse = mse_scores[best_idx]

print("РЕЗУЛЬТАТЫ ПОДБОРА ПАРАМЕТРА p:")
print(f"Оптимальное p = {best_p:.4f}")
print(f"Минимальная MSE = {best_mse:.4f}")

best_p_rounded = round(best_p, 1)
print(f"\nОтвет: {best_p_rounded}")

РЕЗУЛЬТАТЫ ПОДБОРА ПАРАМЕТРА p:
Оптимальное p = 1.0000
Минимальная MSE = 16.0306

Ответ: 1.0
